# Corpus Masterclass 1: A First Look at the State of the Union Corpus

Each code cell below is one teaching beat. Press **Shift+Enter** to run it.

## Today's goal

1. Load every State of the Union address (1790 to 2026) into a pandas DataFrame.
2. Look at individual speeches; summarise the metadata.
3. Plot speech length across two centuries, coloured by party.
4. Build a document-feature matrix: counts of every word in every speech.
5. Draw a word cloud of the most frequent corpus-wide terms.
6. Discover hidden topics with Latent Dirichlet Allocation.
7. Find each speech's most distinctive words.
8. Draw hierarchical-clustering dendrograms; vary the metric and the time slice.
9. Compare word clouds across presidential eras.

In [ ]:
# Libraries used throughout the notebook.
import sotu
import pandas
import numpy
import matplotlib.pyplot as plt
import sklearn.feature_extraction.text
import sklearn.decomposition
import scipy.spatial.distance
import scipy.cluster.hierarchy
import wordcloud
import nltk.stem

print('Libraries imported.')

In [ ]:
# The sotu library exposes every U.S. State of the Union address from 1790 to 2026 as a
# pandas DataFrame. Each row is one speech.
df = sotu.load()
print(f'{len(df)} speeches, {df["year"].min()} to {df["year"].max()}.')
df[['year', 'president', 'party', 'sotu_type']].head(3)

In [ ]:
# Same data, the most recent few addresses.
df[['year', 'president', 'party', 'sotu_type']].tail(3)

In [ ]:
# Washington's first address. The text column holds the full speech as a single string.
first = df.iloc[0]
print(f'{first["year"]} {first["president"]}')
print(first['text'][:600] + ' ...')

In [ ]:
# Washington's second address.
second = df.iloc[1]
print(f'{second["year"]} {second["president"]}')
print(second['text'][:600] + ' ...')

In [ ]:
# Metadata at a glance: column names and types.
df.dtypes

In [ ]:
# How many speeches per party.
df['party'].value_counts()

In [ ]:
# Speech length per address: tokens approximated by whitespace splitting.
df['tokens'] = df['text'].str.split().str.len()
df[['year', 'president', 'tokens']].head(5)

In [ ]:
# Length across two centuries. A single line follows the speeches in year order,
# and the points are coloured by party. Drawing one line per party instead would
# connect speeches across the decades between a party's terms, inventing trends
# that are not there.
by_year = df.sort_values('year')
plt.figure(figsize=(13, 5))
plt.plot(by_year['year'], by_year['tokens'], color='grey', linewidth=0.6, zorder=1)
for party in df['party'].unique():
    party_speeches = df[df['party'] == party]
    plt.scatter(party_speeches['year'], party_speeches['tokens'], s=18, label=party, zorder=2)
plt.xlabel('Year')
plt.ylabel('Tokens')
plt.title('State of the Union address length, 1790 to 2026')
plt.legend(loc='upper left', fontsize=9, ncol=2)
plt.show()

In [ ]:
# A document-feature matrix (DFM) holds the count of every word in every document.
# sklearn.feature_extraction.text.CountVectorizer builds it in one call, with English
# stop-word removal built in.
vectorizer = sklearn.feature_extraction.text.CountVectorizer(
    lowercase=True,
    stop_words='english',
)
dfm = vectorizer.fit_transform(df['text'])
features = vectorizer.get_feature_names_out()
labels = (df['year'].astype(str) + ' ' + df['president']).tolist()

print(f'DFM: {dfm.shape[0]} speeches by {dfm.shape[1]} unique features.')

In [ ]:
# Top 20 features across the whole corpus.
dfm_df = pandas.DataFrame(dfm.toarray(), index=labels, columns=features)
corpus_totals = dfm_df.sum(axis=0).sort_values(ascending=False)
corpus_totals.head(20)

In [ ]:
# Word cloud of the corpus-wide top terms.
wc = wordcloud.WordCloud(
    width=900,
    height=500,
    background_color='white',
    colormap='Dark2',
    random_state=42,
).generate_from_frequencies(corpus_totals.to_dict())

plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('State of the Union: top terms (1790 to 2026)')
plt.show()

In [ ]:
# Why stem at all? In the unstemmed matrix every word form is its own feature, so a
# single idea is split across many columns. The variants of 'nation' below each carry a
# separate count, and stemming rewrites them to a shared root.
# NLTK stemmers: https://www.nltk.org/howto/stem.html
demo_stemmer = nltk.stem.SnowballStemmer('english')
nation_like = []
for word in features:
    if word.startswith('nation'):
        nation_like.append(word)

distinct_roots = []
for word in nation_like:
    root = demo_stemmer.stem(word)
    if root not in distinct_roots:
        distinct_roots.append(root)
    print(f'{word:16s} count {int(corpus_totals[word]):5d}   ->  {root}')

print(f'{len(nation_like)} separate features reduce to {len(distinct_roots)} root(s) after stemming.')

In [ ]:
# Build the stemmed, trimmed matrix that the topic model and the dendrograms will use.
# Stemming merges the variants we just saw into one root. Trimming drops words too rare
# to carry clustering signal. The word cloud and top-feature lists above keep the full,
# unstemmed words.
stemmer = nltk.stem.SnowballStemmer('english')

def stem_text(text):
    roots = []
    for word in text.lower().split():
        roots.append(stemmer.stem(word))
    return ' '.join(roots)

stemmed_docs = []
for text in df['text']:
    stemmed_docs.append(stem_text(text))

stem_vectorizer = sklearn.feature_extraction.text.CountVectorizer(
    stop_words='english',
    min_df=3,
)
stem_counts = stem_vectorizer.fit_transform(stemmed_docs).toarray()
stem_features = stem_vectorizer.get_feature_names_out()

# The other half of dfm_trim: keep only words that occur at least 5 times in total.
column_totals = stem_counts.sum(axis=0)
keep = column_totals >= 5
stem_counts = stem_counts[:, keep]
stem_features = stem_features[keep]
print(f'Stemmed, trimmed: {stem_counts.shape[0]} speeches by {stem_counts.shape[1]} features.')

In [ ]:
# Latent Dirichlet Allocation finds clusters of words that tend to co-occur across
# documents. Each cluster is a topic; each speech is a mixture of topics.
# top_topics runs one model and lists each topic's leading words, tagged with the topic's
# document mass: how many speeches' worth of weight it carries. Written once and reused
# for both runs below.
def top_topics(model):
    document_mass = model.transform(stem_counts).sum(axis=0)
    topics = {}
    for topic_index, weights in enumerate(model.components_):
        top_indices = weights.argsort()[-10:][::-1]
        top_words = []
        for word_index in top_indices:
            top_words.append(stem_features[word_index])
        label = f'Topic {topic_index + 1} (mass {document_mass[topic_index]:.1f})'
        topics[label] = top_words
    return pandas.DataFrame(topics)

In [ ]:
# Run it on the stemmed, trimmed matrix, so the topic words come out as roots. The number
# of topics is a choice. Start with eight, and read the mass tagged on each column.
# scikit-learn LDA: https://scikit-learn.org/stable/modules/decomposition.html#latent-dirichlet-allocation-lda
lda8 = sklearn.decomposition.LatentDirichletAllocation(
    n_components=8,
    random_state=42,
    max_iter=20,
).fit(stem_counts)
top_topics(lda8)

## Eight topics is too many

Two of these topics are identical and own almost no speeches. Eight topics is more than this corpus supports. The surplus topics find no documents, so they collapse onto the same handful of rare words. Choosing the number of topics is a real modelling decision.

Further reading: [scikit-learn's LDA guide](https://scikit-learn.org/stable/modules/decomposition.html#latent-dirichlet-allocation-lda) and [Wallach, Mimno & McCallum (2009), *Rethinking LDA: Why Priors Matter*](https://papers.nips.cc/paper/3854-rethinking-lda-why-priors-matter), which shows how better priors make topic models robust to this choice.

In [ ]:
# Five topics. Every topic now owns real document weight and the duplicates are gone.
lda5 = sklearn.decomposition.LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    max_iter=20,
).fit(stem_counts)
top_topics(lda5)

In [ ]:
# Top 10 words for each of the first three individual speeches.
for row_index in range(3):
    top = dfm_df.iloc[row_index].sort_values(ascending=False).head(10)
    print(f'\n{labels[row_index]}:')
    for word, count in top.items():
        print(f'    {word}: {int(count)}')

In [ ]:
# Row-normalisation of the stemmed, trimmed matrix: divide each row by its total so long
# speeches do not dominate pairwise distances. After this every row sums to 1.
dfm_dense = stem_counts.astype(float)
row_totals = dfm_dense.sum(axis=1, keepdims=True)
row_totals[row_totals == 0] = 1.0
normalised = dfm_dense / row_totals
print(f'Row-sum range: {normalised.sum(axis=1).min():.3f} to {normalised.sum(axis=1).max():.3f}')

In [ ]:
# Helper. Given a matrix slice and labels, draw the hierarchical-clustering
# dendrogram. The metric and title vary; everything else is the same.
def render_dendrogram(matrix, leaf_labels, metric, title, figsize=(14, 6), leaf_font_size=7):
    distances = scipy.spatial.distance.pdist(matrix, metric=metric)
    linkage = scipy.cluster.hierarchy.linkage(distances, method='average')
    plt.figure(figsize=figsize)
    scipy.cluster.hierarchy.dendrogram(
        linkage,
        labels=leaf_labels,
        leaf_rotation=90,
        leaf_font_size=leaf_font_size,
    )
    plt.title(title)
    plt.show()

In [ ]:
# Every speech, Euclidean distance. With 240+ leaves the labels are tiny; the shape
# of the tree is what matters at this scale.
render_dendrogram(
    normalised,
    labels,
    metric='euclidean',
    title='State of the Union, 1790 to 2026: Euclidean distance',
    figsize=(22, 7),
    leaf_font_size=4,
)

In [ ]:
# Modern subset, Euclidean distance.
modern_mask = (df['year'] >= 1977).to_numpy()
modern_matrix = normalised[modern_mask]
modern_labels = []
for label, keep in zip(labels, modern_mask):
    if keep:
        modern_labels.append(label)

render_dendrogram(
    modern_matrix,
    modern_labels,
    metric='euclidean',
    title='Modern State of the Union (1977+): Euclidean distance',
)

In [ ]:
# Same modern subset, cosine distance. Cosine measures the angle between two count
# vectors and ignores their magnitude.
render_dendrogram(
    modern_matrix,
    modern_labels,
    metric='cosine',
    title='Modern State of the Union (1977+): cosine distance',
)

In [ ]:
# Founders era, Euclidean distance.
founders_mask = (df['year'] < 1850).to_numpy()
founders_matrix = normalised[founders_mask]
founders_labels = []
for label, keep in zip(labels, founders_mask):
    if keep:
        founders_labels.append(label)

render_dendrogram(
    founders_matrix,
    founders_labels,
    metric='euclidean',
    title='Founders-era State of the Union (pre-1850): Euclidean distance',
    figsize=(18, 6),
    leaf_font_size=6,
)

In [ ]:
# Founders era, cosine distance. Compare the four dendrograms above and ask which
# pairs of speeches consistently cluster together regardless of metric and time slice.
render_dendrogram(
    founders_matrix,
    founders_labels,
    metric='cosine',
    title='Founders-era State of the Union (pre-1850): cosine distance',
    figsize=(18, 6),
    leaf_font_size=6,
)

In [ ]:
# Helper for the comparison-wordcloud beats. Takes one or more speaker names and lays
# their first few addresses out as a grid of clouds.
def comparison_clouds(speakers, take=4):
    if isinstance(speakers, str):
        speakers = [speakers]
    rows = []
    for name in speakers:
        for _, row in df[df['president'] == name].head(take).iterrows():
            rows.append(row)
    if not rows:
        print(f'No matches for {speakers!r}')
        return
    cols = 2 if len(speakers) == 1 else 4
    nrows = int(numpy.ceil(len(rows) / cols))
    fig, axes = plt.subplots(nrows, cols, figsize=(11, 3.2 * nrows))
    flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, row in zip(flat, rows):
        wc_local = wordcloud.WordCloud(
            width=500,
            height=350,
            background_color='white',
            colormap='tab10',
            random_state=42,
            stopwords=wordcloud.STOPWORDS,
        ).generate(row['text'])
        ax.imshow(wc_local, interpolation='bilinear')
        ax.set_title(f'{row["year"]} {row["president"]}', fontsize=10)
        ax.axis('off')
    for ax in flat[len(rows):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Washington's first four addresses.
comparison_clouds('Washington')

In [ ]:
# Adams and Jefferson side by side.
comparison_clouds(['Adams', 'Jefferson'])

In [ ]:
# Obama and Trump side by side.
comparison_clouds(['Obama', 'Trump'])

## What we've covered

**Python:** library imports, pandas DataFrame inspection and column assignment, boolean masking, `for` loops, defining functions with default arguments, f-strings, matplotlib subplots.

**Corpus linguistics:** loading a built-in corpus; document-feature matrix; English stop-word filtering; corpus-wide vs per-document top features; word clouds; stemming to merge word variants; Latent Dirichlet Allocation for topic discovery and the effect of choosing the number of topics; hierarchical clustering with Euclidean and cosine distance; dendrograms; comparison word clouds across presidential eras.

## Read more

- **NLTK Book**, Chapter 1: [https://www.nltk.org/book/ch01.html](https://www.nltk.org/book/ch01.html).
- **scikit-learn user guide on text feature extraction**: [https://scikit-learn.org/stable/modules/feature_extraction.html](https://scikit-learn.org/stable/modules/feature_extraction.html).
- **scikit-learn user guide on Latent Dirichlet Allocation**: [https://scikit-learn.org/stable/modules/decomposition.html#latent-dirichlet-allocation-lda](https://scikit-learn.org/stable/modules/decomposition.html#latent-dirichlet-allocation-lda).
- **scipy hierarchical clustering**: [https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html](https://docs.scipy.org/doc/scipy/reference/cluster.hierarchy.html).
- **matplotlib pyplot tutorial**: [https://matplotlib.org/stable/tutorials/pyplot.html](https://matplotlib.org/stable/tutorials/pyplot.html).
- **wordcloud README**: [https://github.com/amueller/word_cloud](https://github.com/amueller/word_cloud).
- **matplotlib named colormaps**: [https://matplotlib.org/stable/tutorials/colors/colormaps.html](https://matplotlib.org/stable/tutorials/colors/colormaps.html).